# Conteo de registros H y M

Sistemas de Big Data 5074 · UT1.2 · Diapositivas 1–11.

Versión para completar. Puedes trabajar en este cuaderno en lugar de `alumno.py`. Sustituye los `NotImplementedError` por tu código. Las celdas con tareas pendientes se detendrán hasta que las completes.

Solo biblioteca estándar de Python 3.10 o posterior. Los datos están incluidos. No requiere instalaciones ni archivos externos. Puedes subir este `.ipynb` a Colab. Los ejemplos MapReduce se simulan localmente y en serie, sin ejecutar Hadoop.


# Conteo de registros por categoría H/M

El objetivo consiste en contar cuántos registros contienen la etiqueta H y cuántos la etiqueta M. La categoría se lee del último campo. El nombre no determina esa etiqueta.

Trabaja en `alumno.py`. Necesitas Python 3.10 o posterior y su biblioteca estándar. Los imports y los datos ya están preparados. Sustituye cada `NotImplementedError` por tu implementación. Ver ese error antes de completar una función es normal en esta plantilla.

## Datos y alcance

Hay tres ficheros con tres registros cada uno. Los tres registros de `1800.txt` reproducen el ejemplo visible de la diapositiva 3. Los otros seis registros son ficticios, creados para esta práctica. El resultado de los nueve registros corresponde a esta práctica, no a un conjunto completo mostrado en la presentación.

`FICHEROS` contiene los datos dentro del propio programa. Los archivos de `datos/` son su copia en UTF-8 con saltos de línea LF. No necesitas leerlos del disco para ejecutar el programa.

Cada fichero representa un split lógico didáctico. El recorrido se ejecutará de forma local y serial. H/M son las etiquetas de este conjunto didáctico. Contamos apariciones de registros, sin deduplicar personas entre ficheros.

## Trabajo paso a paso

1. Lee los nueve registros y calcula a mano el resultado esperado. Anota qué campo debe ser la clave intermedia para poder sumar por categoría.
2. Completa `map_registro`. Separa los campos y comprueba que hay un nombre no vacío, una sola coma y una categoría válida. Emite un par por registro. Pistas: `rpartition` y `strip`.
3. Completa `shuffle`. Recorre los pares emitidos y construye una lista de valores por clave. Conserva todos los valores repetidos. Pista: `defaultdict`.
4. Completa `reduce_categoria`. Escribe un bucle que acumule los valores de un grupo y devuelve el par de salida. Después contrasta tu acumulación con `sum`.
5. Completa `ejecutar_mapreduce`. Escribe los bucles que recorren los ficheros y sus líneas, recoge las emisiones, agrupa y reduce. Si un registro falla, informa del fichero y la línea. Pistas: `splitlines` y `enumerate`.
6. Completa `conteo_secuencial` con un recorrido directo que actualice los contadores sin construir emisiones ni grupos.
7. Completa `mostrar_demostracion`: muestra las nueve emisiones Map, los grupos y los dos resultados Reduce. Compara los métodos y comprueba que no se pierde ningún registro.

## Evidencias que debes obtener

- Nueve emisiones Map, una por registro.
- Dos grupos, uno por categoría, con todos sus valores.
- Un diccionario final con el conteo que hayas calculado para cada categoría.
- El mismo resultado mediante el conteo secuencial.
- Igualdad entre registros, emisiones Map y valores agrupados. Comprueba que `H + M = 9` usando los conteos que hayas obtenido.

Un registro vacío, uno sin coma, uno sin nombre o uno con categoría distinta de H/M debe provocar un error claro. No lo ignores silenciosamente. Los saltos LF que cierran los registros no forman un registro adicional.

## Relación con la presentación

Las diapositivas 7–9 usan el nombre como clave intermedia. La diapositiva 10 compara después la clave con H/M. Explica por qué esas dos decisiones no encajan. Para este objetivo la clave intermedia es la categoría, que es precisamente aquello por lo que se quiere contar.

## Qué representa la simulación

Las funciones muestran la transformación Map, la agrupación por clave y la reducción. El programa no ejecuta Hadoop ni reproduce HDFS, la transferencia por red, collect, spill, merge, planificación o tolerancia a fallos. No se deben extraer conclusiones de rendimiento distribuido a partir de esta ejecución local.


## Código 1 Presentación del bloque


In [10]:
"""Plantilla intermedia: conteo de registros por categoría H/M.

Python 3.10 o posterior. Solo biblioteca estándar.
Completa las funciones sustituyendo cada NotImplementedError.
Los tres primeros registros son los del ejemplo de la diapositiva 3.
Los otros seis son ficticios y se han añadido para esta práctica.
Cada fichero representa un split lógico didáctico. La ejecución es serial
y local. Esta plantilla no ejecuta Hadoop.
"""


'Plantilla intermedia: conteo de registros por categoría H/M.\n\nPython 3.10 o posterior. Solo biblioteca estándar.\nCompleta las funciones sustituyendo cada NotImplementedError.\nLos tres primeros registros son los del ejemplo de la diapositiva 3.\nLos otros seis son ficticios y se han añadido para esta práctica.\nCada fichero representa un split lógico didáctico. La ejecución es serial\ny local. Esta plantilla no ejecuta Hadoop.\n'

## Código 2 Imports y datos preparados


In [11]:
from collections import defaultdict


FICHEROS: dict[str, str] = {
    "1800.txt": (
        "Manuel García García, H\n"
        "Juan Belmonte Pérez, H\n"
        "María Selas Rodríguez, M\n"
    ),
    "1801.txt": (
        "Lucía Martín Pérez, M\n"
        "Diego Ramos Díaz, H\n"
        "Elena Suárez León, M\n"
    ),
    "1802.txt": (
        "Pablo Torres Gil, H\n"
        "Nora Medina Ruiz, M\n"
        "Iván Cabrera Soto, H\n"
    ),
}

CATEGORIAS = ("H", "M")


## Código 3 Map: parseo, validación y emisión


In [12]:
def map_registro(linea: str) -> tuple[str, int]:
    """Recibe una línea sin terminador y emite (categoría, 1).

    TODO: separa nombre y categoría. Rechaza nombres vacíos, registros
    sin una única coma, saltos de línea internos y categorías inválidas.
    Lee H/M del campo final. No deduzcas la categoría a partir del nombre.
    Pista: investiga rpartition y strip.
    """
    # Comprobar si hay saltos de línea internos
    if "\n" in linea or "\r" in linea:
        raise ValueError("Se detectó un salto de línea interno en el registro.")

    # Separar usando rpartition para aislar el último campo (la categoría)
    nombre, separador, categoria = linea.rpartition(",")

    if separador != ",":
        raise ValueError(f"El registro no contiene una única coma válida: '{linea}'")

    nombre = nombre.strip()
    categoria = categoria.strip()

    # Validar que el nombre no esté vacío
    if not nombre:
        raise ValueError("El nombre del registro está vacío.")

    # Validar que la categoría sea estrictamente H o M
    if categoria not in CATEGORIAS:
        raise ValueError(f"Categoría inválida '{categoria}'. Debe ser H o M.")

    return (categoria, 1)

## Código 4 Shuffle: agrupación sin perder repeticiones


In [13]:
def shuffle(pares: list[tuple[str, int]]) -> dict[str, list[int]]:
    """Recibe emisiones Map y devuelve una lista de valores por clave.

    TODO: escribe el recorrido de pares y la construcción de los grupos.
    Cada 1 debe conservarse, aunque otros valores sean iguales.
    Pista: defaultdict puede facilitar la creación de listas vacías.
    """
    grupos = defaultdict(list)
    for clave, valor in pares:
        grupos[clave].append(valor)
    return dict(grupos)  # Opcional: convertir de vuelta a diccionario estándar si se prefiere

## Código 5 Reduce: cálculo por categoría


In [14]:
def reduce_categoria(clave: str, valores: list[int]) -> tuple[str, int]:
    """Devuelve (categoría, total) para un único grupo.

    TODO: escribe la acumulación de los valores y devuelve el par de salida.
    Pista: resuélvelo primero con un bucle. Después puedes compararlo con sum.
    """
    total = 0
    for valor in valores:
        total += valor

    # O alternativamente usando sum(valores)
    return (clave, total)

## Código 6 Coordinación de las fases


In [15]:
def ejecutar_mapreduce(ficheros: dict[str, str]) -> dict[str, int]:
    """Devuelve el conteo de las categorías presentes en los registros.

    TODO: recorre los ficheros y líneas, llama a Map y recoge sus pares.
    Después agrupa y aplica Reduce a cada clave. Construye el resultado.
    Los errores deben indicar fichero y número de línea, sin ignorarlos.
    Pista: splitlines, enumerate y los contratos de las funciones anteriores.
    """
    pares_emitidos = []

    # 1. Recorrer ficheros y líneas, emitiendo pares (Map) con control de errores por línea
    for nombre_fichero, contenido in ficheros.items():
        for num_linea, linea in enumerate(contenido.splitlines(), start=1):
            if linea.strip():
                try:
                    par = map_registro(linea)
                    pares_emitidos.append(par)
                except Exception as e:
                    raise ValueError(f"Error en el fichero '{nombre_fichero}', línea {num_linea} ('{linea}'): {e}")

    # 2. Agrupar los pares (Shuffle)
    grupos = shuffle(pares_emitidos)

    # 3. Reducir cada grupo (Reduce)
    resultado = {}
    for clave, valores in grupos.items():
        cat, total = reduce_categoria(clave, valores)
        resultado[cat] = total

    return resultado

## Código 7 Contraste secuencial


In [16]:
def conteo_secuencial(ficheros: dict[str, str]) -> dict[str, int]:
    """Cuenta directamente los registros H/M sin crear listas de emisiones.

    TODO: recorre y valida los datos. Actualiza un contador por categoría.
    Este recorrido permitirá contrastar tu resultado MapReduce.
    """
    conteo = {"H": 0, "M": 0}

    for nombre_fichero, contenido in ficheros.items():
        for num_linea, linea in enumerate(contenido.splitlines(), start=1):
            if linea.strip():
                # Reutilizamos map_registro para aprovechar las validaciones de formato
                categoria, _ = map_registro(linea)
                conteo[categoria] += 1

    return conteo

## Código 8 Mostrar las fases y comprobar el resultado


In [17]:
def mostrar_demostracion() -> None:
    """Construye una traza de los datos preparados.

    TODO: muestra una emisión por registro, los grupos y cada salida Reduce.
    Ejecuta ambos métodos y comprueba su igualdad. Comprueba también que el
    número de registros, emisiones y valores agrupados coincide con H + M.
    Calcula los conteos de H y M. Su suma debe ser igual a los 9 registros.
    """
    # 1. Recopilar todas las emisiones Map de los 9 registros
    emisiones = []
    total_registros = 0
    for nombre_fichero, contenido in FICHEROS.items():
        for linea in contenido.splitlines():
            if linea.strip():
                total_registros += 1
                par = map_registro(linea)
                emisiones.append(par)
                print(f"Emisión Map: {par}")

    print(f"\nTotal registros procesados: {total_registros}")
    print(f"Total emisiones Map: {len(emisiones)}")

    # 2. Aplicar Shuffle para agrupar
    grupos = shuffle(emisiones)
    print(f"\nGrupos obtenidos (Shuffle): {grupos}")

    # Comprobación de valores agrupados totales
    total_valores_agrupados = sum(len(valores) for valores in grupos.values())
    print(f"Total valores en grupos: {total_valores_agrupados}")

    # 3. Aplicar Reduce y obtener salidas
    resultados_mr = {}
    print("\nSalidas Reduce:")
    for clave, valores in grupos.items():
        cat, total = reduce_categoria(clave, valores)
        resultados_mr[cat] = total
        print(f"Categoría '{cat}': {total}")

    # 4. Ejecutar método secuencial para contraste
    resultados_sec = conteo_secuencial(FICHEROS)
    print(f"\nResultado Conteo Secuencial: {resultados_sec}")

    # 5. Comprobaciones e igualdades
    assert resultados_mr == resultados_sec, "El resultado MapReduce y el secuencial deben ser idénticos."

    conteo_h = resultados_mr.get("H", 0)
    conteo_m = resultados_mr.get("M", 0)

    print(f"\nConteo H: {conteo_h}, Conteo M: {conteo_m}")
    print(f"Suma H + M = {conteo_h + conteo_m}")

    # Comprobación final solicitada
    assert total_registros == len(emisiones) == total_valores_agrupados == (conteo_h + conteo_m) == 9, \
        "Error en la comprobación de integridad de registros."

    print("\n¡Demostración y comprobaciones superadas con éxito!")

## Código 9 Ejecutar tu demostración


In [18]:
if __name__ == "__main__":
    mostrar_demostracion()


Emisión Map: ('H', 1)
Emisión Map: ('H', 1)
Emisión Map: ('M', 1)
Emisión Map: ('M', 1)
Emisión Map: ('H', 1)
Emisión Map: ('M', 1)
Emisión Map: ('H', 1)
Emisión Map: ('M', 1)
Emisión Map: ('H', 1)

Total registros procesados: 9
Total emisiones Map: 9

Grupos obtenidos (Shuffle): {'H': [1, 1, 1, 1, 1], 'M': [1, 1, 1, 1]}
Total valores en grupos: 9

Salidas Reduce:
Categoría 'H': 5
Categoría 'M': 4

Resultado Conteo Secuencial: {'H': 5, 'M': 4}

Conteo H: 5, Conteo M: 4
Suma H + M = 9

¡Demostración y comprobaciones superadas con éxito!


## Explicación de tus resultados

Anota aquí tus conclusiones y las respuestas que pide el guion.




### 1. Resumen de la implementación
  * **`map_registro`**: Valida que cada línea contenga una única coma mediante `rpartition`, que el nombre no esté vacío tras aplicar `strip`, que no existan saltos de línea internos y que la categoría final sea estrictamente `H` o `M`, emitiendo el par `(categoría, 1)`.
  * **`shuffle`**: Recorre las emisiones generadas y agrupa los valores por clave utilizando `defaultdict(list)`, conservando todas las repeticiones sin perder ningún registro.
  * **`reduce_categoria`**: Acumula los unos de cada grupo mediante un bucle (contrastable con `sum`) y devuelve la tupla `(categoría, total)`.
  * **`ejecutar_mapreduce`**: Coordina el recorrido de los tres ficheros (`1800.txt`, `1801.txt`, `1802.txt`), captura cualquier error de formato indicando el fichero y número de línea exacto con `enumerate`, agrupa las emisiones y reduce cada categoría.
  * **`conteo_secuencial`**: Realiza un recorrido directo validando los registros y sumando los contadores sin crear listas de emisiones ni grupos intermedios, sirviendo como referencia de contraste.
  * **`mostrar_demostracion`**: Muestra la traza completa (las 9 emisiones Map, los 2 grupos generados y las salidas Reduce) y verifica la igualdad entre ambos métodos, comprobando que $H + M = 9$ registros totales.


### 2. Resultados obtenidos
  * **Emisiones Map**: 9 pares emitidos (uno por cada registro válido).
  * **Grupos (Shuffle)**: `{'H': [1, 1, 1, 1, 1], 'M': [1, 1, 1, 1]}`.
  * **Conteo final (MapReduce y Secuencial)**: `{'H': 5, 'M': 4}`, cumpliéndose que $5 + 4 = 9$ registros sin pérdida de datos.


### 3. Relación con la presentación (Diapositivas 7–10)
  En las diapositivas 7–9 se utiliza el **nombre** como clave intermedia, pero en la diapositiva 10 se intenta comparar esa clave con las categorías `H`/`M`. Estas dos decisiones no encajan porque la fase *Shuffle* agrupa exactamente por la clave emitida en el *Map*: si se emite el nombre, el *Reduce* recibirá grupos por persona y no por sexo/categoría. Para contar cuántos registros pertenecen a `H` y cuántos a `M`, la clave intermedia debe ser necesariamente la **categoría**.


### 4. Alcance de la simulación
  Esta ejecución es puramente didáctica, local y en serie sobre memoria. Simula la lógica funcional de transformación (*Map*), agrupación por clave (*Shuffle*) y agregación (*Reduce*), pero **no ejecuta Hadoop** ni reproduce almacenamiento distribuido (HDFS), transferencia por red, fases internas de *collect/spill/merge*, planificación de tareas ni tolerancia a fallos, por lo que no permite extraer métricas de rendimiento distribuido.